# ADTC - Balanced QLoRA Re-train (Option B)
Goal: fix the 81%-Hausa imbalance + ARC forgetting from the first run.

Data: AfriQA (QA ha/yo/ig/sw, train+validation), Belebele (MCQA 122 langs), AfroMLMU (exam MCQA) + Masakhane african-ultrachat (multi-turn chat) for ALL five languages, plus ADTC Yoruba mix (43k chat) and an English WAEC integrated-science set. Each language capped at MAX_PER_LANG so no single language dominates.

Train: Qwen2.5-0.5B-Instruct + unsloth QLoRA (low LR, small rank, 2 epochs, cosine schedule) to curb forgetting.

Eval (honest, held-out): ARC-Easy on the test split (n=ARC_N=500, robust A-D extraction) measures general reasoning so we can catch forgetting; AfriQA per-language on its held-out test split (n=300) measures WAEC-like QA. Eval uses test splits that were NOT in training. Writes summary.json (base / fine_tuned / verdict / notes / meta) for the auto-check.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q','-U','unsloth','datasets','transformers','accelerate','peft','bitsandbytes','trl','huggingface_hub'])
print('installs done')


In [ ]:
import torch, random, numpy as np
from datasets import load_dataset, Dataset
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_PER_LANG = 4000
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
FLORES = {'en':'eng','yo':'yor','ha':'hau','sw':'swa','ig':'ibo'}
AFRIQA_LANG = {'yo':'yor','ha':'hau','sw':'swa','ig':'ibo'}
LANG_DISPLAY = {'en':'English','yo':'Yoruba','ha':'Hausa','sw':'Swahili','ig':'Igbo'}

def qa_messages(question, answer):
    return [{'role':'user','content':str(question)},
            {'role':'assistant','content':str(answer)}]

def load_afriqa(lang):
    try:
        ds = load_dataset('masakhane/afriqa', lang, split='train+validation')
        out=[]
        for r in ds:
            a=r['answers']
            a=a[0] if isinstance(a,list) and a else a
            out.append({'messages':qa_messages(r['question'],a),'lang':lang})
        return out
    except Exception as e:
        print('afriqa',lang,'skip',e); return []

def load_belebele(fl):
    try:
        ds = load_dataset('facebook/belebele', f'{fl}_Latn', split='test')
        L=['A','B','C','D']; out=[]
        for r in ds:
            opts=[r['mc_answer1'],r['mc_answer2'],r['mc_answer3'],r['mc_answer4']]
            p=f"{r['flores_passage']}\n\nQuestion: {r['question']}\n"+''.join(f'{L[i]}) {opts[i]}\n' for i in range(4))+'Answer:'
            out.append({'messages':qa_messages(p,L[int(r['correct_answer_num'])-1]),'lang':fl})
        return out
    except Exception as e:
        print('belebele',fl,'skip',e); return []

def load_afrimmlu(lang):
    try:
        ds = load_dataset('masakhane/afrimmlu', lang, split='test')
        L=['A','B','C','D']; out=[]
        for r in ds:
            ch=r['choices']
            p=f"Subject: {r.get('subject','')}\nQuestion: {r['question']}\n"+''.join(f'{L[i]}) {ch[i]}\n' for i in range(len(ch)))+'Answer:'
            out.append({'messages':qa_messages(p,L[int(r['answer'])]),'lang':lang})
        return out
    except Exception as e:
        print('afrimmlu',lang,'skip',e); return []

def load_adtc_yoruba():
    try:
        ds = load_dataset('1nnocent/adtc-agri-yoruba-training-mix', split='train')
        out=[]
        for r in ds:
            m=r.get('messages')
            if isinstance(m,list) and m: out.append({'messages':m,'lang':'yor'})
        return out
    except Exception as e:
        print('adtc-yoruba skip',e); return []

def load_hausa_extra():
    out=[]
    for src in ['honourjesus/nllb-hausa-waec-translations','Wayazi/adtc-healthcare-dataset']:
        try:
            ds=load_dataset(src, split='train')
            for r in ds:
                txt=r.get('text') or r.get('messages') or None
                if isinstance(txt,list): out.append({'messages':txt,'lang':'hau'})
                elif isinstance(txt,str): out.append({'messages':qa_messages(txt,txt),'lang':'hau'})
        except Exception as e:
            print('hausa-src',src,'skip',e)
    return out

def load_ultrachat(lang):
    try:
        ds = load_dataset('masakhane/african-ultrachat', split='train')
        out=[]
        for r in ds:
            if str(r.get('language','')).strip().lower()==LANG_DISPLAY[lang].lower():
                m=r.get('messages')
                if isinstance(m,list) and m: out.append({'messages':m,'lang':lang})
        return out
    except Exception as e:
        print('ultrachat',lang,'skip',e); return []

def load_waec_science():
    try:
        ds = load_dataset('worldboss/waec-integrated-science-2007', split='train')
        out=[]
        for r in ds:
            q=r.get('Question'); a=r.get('Answer')
            if q and a:
                out.append({'messages':qa_messages(str(q).strip(), str(a).strip()),'lang':'eng'})
        return out
    except Exception as e:
        print('waec-science skip',e); return []

collections = {
  'yor': load_afriqa('yor') + load_belebele('yor') + load_afrimmlu('yor') + load_adtc_yoruba() + load_ultrachat('yor'),
  'hau': load_afriqa('hau') + load_belebele('hau') + load_afrimmlu('hau') + load_hausa_extra() + load_ultrachat('hau'),
  'ibo': load_afriqa('ibo') + load_belebele('ibo') + load_afrimmlu('ibo') + load_ultrachat('ibo'),
  'swa': load_afriqa('swa') + load_belebele('swa') + load_afrimmlu('swa') + load_ultrachat('swa'),
  'eng': load_belebele('eng') + load_afrimmlu('eng') + load_waec_science() + load_ultrachat('eng'),
}
capped=[]
for lang,rows in collections.items():
    random.shuffle(rows); rows=rows[:MAX_PER_LANG]; capped+=rows
random.shuffle(capped)
from collections import Counter
lang_counts = Counter(r['lang'] for r in capped)
data = Dataset.from_list(capped)
print('TOTAL ROWS:', len(data))
for lang in sorted(lang_counts, key=lambda l: -lang_counts[l]):
    pct = lang_counts[lang]/len(data)*100
    print(f'  {lang} ({LANG_DISPLAY.get(lang,lang)}): {lang_counts[lang]:5d}  ({pct:4.1f}%)')
print('balance check ->', {l: lang_counts[l] for l in sorted(lang_counts)})


In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
max_seq = 1536  # bumped from 1024 to reduce truncation of long chat/Belebele rows
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL, max_seq_length=max_seq, dtype=torch.float16, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    bias='none', use_gradient_checkpointing='unsloth')
print('model + LoRA ready')

def fmt(ex):
    return {'text': tokenizer.apply_chat_template(ex['messages'], tokenize=False, add_generation_prompt=False)}
data = data.map(fmt)
print('formatted', len(data))


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=data, dataset_text_field='text',
    args=TrainingArguments(
        per_device_train_batch_size=4, gradient_accumulation_steps=4,
        warmup_steps=10, num_train_epochs=2, learning_rate=1e-4,
        fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
        logging_steps=10, output_dir='outputs', optim='adamw_8bit',
        lr_scheduler_type='cosine', report_to='none', seed=SEED))
trainer.train()
model.save_pretrained('adtc_qwen_lora')
tokenizer.save_pretrained('adtc_qwen_lora')
print('trained + saved')


In [ ]:
import re, json, gc
ARC_N=500      # ARC-Easy eval sample size (set None to use full test set for max stability)
AFRIQA_N=300   # per-language AfriQA eval sample size

def extract_letter(text):
    m = re.search(r'(?:answer\s*is\s*|\(?)([A-D])(?:\s*\)|\b)', text, re.I)
    if m: return m.group(1).upper()
    for L in ['A','B','C','D']:
        if re.search(r'\b'+L+r'\b', text): return L
    return ''

def gen_answer(prompt, mdl, tok, max_new=80):
    ids = tok.apply_chat_template([{'role':'user','content':prompt}], tokenize=True,
                                  add_generation_prompt=True, return_tensors='pt').to(mdl.device)
    out = mdl.generate(ids, max_new_tokens=max_new, do_sample=False)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

def eval_arc(mdl, tok, n=ARC_N):
    ds = load_dataset('allenai/ai2_arc','ARC-Easy', split='test')
    if n and n < len(ds): ds = ds.shuffle(seed=SEED).select(range(n))
    L=['A','B','C','D']; c=0
    for r in ds:
        q=r['question']['stem']; opts=r['question']['choices']
        p='Question: '+q+'\n'+''.join(f'{L[i]}) {o["text"]}\n' for i,o in enumerate(opts))+'Answer:'
        a=extract_letter(gen_answer(p,mdl,tok))
        gold=L[[o['label'] for o in opts].index(r['answerKey'])]
        if a==gold: c+=1
    return c/len(ds)

def eval_afriqa(mdl, tok, lang, n=AFRIQA_N):
    ds = load_dataset('masakhane/afriqa', lang, split='test')
    if n and n < len(ds): ds = ds.shuffle(seed=SEED).select(range(n))
    c=0
    for r in ds:
        g=r['answers']; g=(g[0] if isinstance(g,list) and g else g)
        a=gen_answer(r['question'],mdl,tok,max_new=80).lower()
        if str(g).strip().lower() in a or a in str(g).strip().lower(): c+=1
    return c/len(ds)

# ---- BASE MODEL EVAL (plain, no LoRA) ----
base_model, base_tok = FastLanguageModel.from_pretrained(model_name=MODEL, max_seq_length=max_seq, dtype=torch.float16, load_in_4bit=True)
base = {'arc':{'arc_easy_gen_acc':eval_arc(base_model,base_tok),'n':ARC_N},
        'waec_by_lang':{l:round(eval_afriqa(base_model,base_tok,AFRIQA_LANG[l]),3) for l in ['yo','ha','sw','ig']},
        'waec_overall':None}
base['waec_by_lang']['en']=None
base['waec_overall']=round(sum(v for v in base['waec_by_lang'].values() if v is not None)/4,3)
print('BASE:', base)
del base_model; gc.collect(); torch.cuda.empty_cache()


In [ ]:
# ---- FINE-TUNED EVAL (LoRA model still in memory) ----
ft = {'arc':{'arc_easy_gen_acc':eval_arc(model,tokenizer),'n':ARC_N},
      'waec_by_lang':{l:round(eval_afriqa(model,tokenizer,AFRIQA_LANG[l]),3) for l in ['yo','ha','sw','ig']},
      'waec_overall':None}
ft['waec_by_lang']['en']=None
ft['waec_overall']=round(sum(v for v in ft['waec_by_lang'].values() if v is not None)/4,3)

verdict={'arc_improved': ft['arc']['arc_easy_gen_acc'] > base['arc']['arc_easy_gen_acc'],
         'waec_improved': ft['waec_overall'] > base['waec_overall']}

# honest flags (no cherry-pick): per-language forgetting / regressions
notes=[]
if not verdict['arc_improved']:
    notes.append(f"ARC-Easy regressed {base['arc']['arc_easy_gen_acc']:.3f} -> {ft['arc']['arc_easy_gen_acc']:.3f} (forgetting watch)")
for l in ['yo','ha','sw','ig']:
    b=base['waec_by_lang'][l]; f=ft['waec_by_lang'][l]
    if b is not None and f is not None and f < b - 0.02:
        notes.append(f"{l} WAEC score dropped {b:.3f} -> {f:.3f}")
if not notes:
    notes.append("no per-language regression >2pp detected")

meta={'model':MODEL,'max_per_lang':MAX_PER_LANG,'epochs':2,'lr':1e-4,'r':16,
      'max_seq':max_seq,'arc_n':ARC_N,'afriqa_n':AFRIQA_N,'seed':SEED,
      'data_balance':{l:int(lang_counts[l]) for l in sorted(lang_counts)}}

summary={'base':base,'fine_tuned':ft,'verdict':verdict,'notes':notes,'meta':meta}
import json as _j
_j.dump(summary, open('summary.json','w'), indent=2)
print(_j.dumps(summary, indent=2))
print('Wrote summary.json - drop it into ~/adtc_results or Download/ for the auto-check, or just read above.')
